# Notebook 4 — Generation & Evaluation (REVISED)

Key changes from original:
1. **CrossAttentionBindingScorer**: uses `ab_proj` (full seqcoding 480d), no `forward_contrastive`.
2. **Phase 1 ODE**: guidance uses `forward_score` (cross-attention), not contrastive cosine.
3. **Phase 2 steering**: `steer_logits()` — joint bonus: PLS distance (developability) + binding score from **one** seqcoding call per trial token.
4. **Iterative rounds**: 5 rounds of steering with population update and tightening binding threshold.
5. **compare_df** includes pre-computed binding scores and all VH columns for evaluation notebook.

In [3]:
!pip install -q ablang2 torchcfm torchdiffeq scikit-learn scipy matplotlib
!pip install -q seaborn joblib fair-esm anarci requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 129.3 MB/s eta 0:00:00


In [4]:
import os, re, warnings, json, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy.stats import spearmanr, pearsonr, ttest_rel, ks_2samp
from scipy.spatial.distance import cdist as scipy_cdist
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold
import joblib, ast

import esm
import ablang2

from torchcfm.conditional_flow_matching import ConditionalFlowMatcher
from torchdiffeq import odeint

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
torch.manual_seed(42)
np.random.seed(42)

Device: cuda


In [5]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/2026 Spring/BMI 702/project'
EVAL_DIR    = f'{PROJECT_DIR}/eval_inputs'
DATA_PATH   = f'{PROJECT_DIR}/GDPa1_v1.2_20250814.csv'

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(EVAL_DIR,    exist_ok=True)

CACHE = lambda name: f'{PROJECT_DIR}/{name}'

COL_VH    = 'vh_protein_sequence'
COL_VL    = 'vl_protein_sequence'
COL_HIC   = 'HIC'
COL_SINS  = 'AC-SINS_pH7.4'
COL_FOLD  = 'hierarchical_cluster_fold'
COL_AHO_H = 'heavy_aligned_aho'
COL_NAME  = 'antibody_name'

TEST_FOLD       = 1
N_FLOW_SAMPLES  = 50
N_ODE_STEPS     = 100
M_SEQUENCES     = 20
TEMPERATURE     = 1.0
TOP_K_SELECT    = 5
FLOW_EPOCHS     = 1000
CURRENT_SIGMA   = 0.5
PLS_COMPONENTS  = 10
AA_VOCAB        = list('ACDEFGHIKLMNPQRSTVWY')
AA_TO_IDX       = {aa: i for i, aa in enumerate(AA_VOCAB)}
ABLANG_MASK     = '*'

SCORER_HIDDEN   = 128
SCORER_HEADS    = 4
ANTIGEN_MAX_LEN = 512
GUIDANCE_LAMBDA = 0.5
ESM2_DIM        = 1280

# Phase 2 hyperparameters
N_STEER_ROUNDS    = 5
TOP_K_STEER       = 5
TOP_K_TOKENS      = 5
GUIDANCE_STRENGTH = 2.0
ALPHA_DEV         = 1.0
BETA_BIND         = 1.0
BIND_SLACK_INIT   = 0.5

# Antibodies without human antigen — excluded from binding evaluation
NO_ANTIGEN_ABS = {'panobacumab', 'racotumomab'}

MEAN_CACHE      = f'{EVAL_DIR}/antigen_esm2_embs.npy'
RESID_CACHE     = f'{EVAL_DIR}/antigen_esm2_resid.npy'
MASK_CACHE      = f'{EVAL_DIR}/antigen_valid_mask.npy'
RESID_PAD_CACHE = f'{EVAL_DIR}/antigen_resid_pad_mask.npy'

print('Configuration complete.')

Mounted at /content/drive
Configuration complete.


# ─────────────────────────────────────────────────────────────────────────────
# Section 1 — Reload all artifacts
# ─────────────────────────────────────────────────────────────────────────────

In [6]:
df          = pd.read_csv(f'{PROJECT_DIR}/clinical_mabs_with_antigens.csv')
oracle_hic  = joblib.load(CACHE('oracle_hic.pkl'))
oracle_sins = joblib.load(CACHE('oracle_sins.pkl'))
full_scaler = joblib.load(CACHE('full_scaler.pkl'))
pls         = joblib.load(CACHE('pls_model.pkl'))
pls_scaler  = joblib.load(CACHE('pls_scaler.pkl'))
dev_scaler  = joblib.load(CACHE('dev_scaler.pkl'))

data = np.load(CACHE('flow_model_inputs.npz'), allow_pickle=True)
X_pls_tr_sc  = data['X_pls_tr_sc']
X_pls_te_sc  = data['X_pls_te_sc']
X_fw_tr      = data['X_fw_tr']
X_fw_te      = data['X_fw_te']
y_hic_train  = data['y_hic_train']
y_sins_train = data['y_sins_train']
test_idx     = data['test_idx'].tolist()
train_idx    = data['train_idx'].tolist()
PLS_DIM      = X_pls_tr_sc.shape[1]
FW_DIM       = X_fw_tr.shape[1]

for col in ['h_cdr3_idx', 'h_fw_idx']:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

if hasattr(pls, 'x_mean_'):
    pls_x_mean = pls.x_mean_; pls_x_loadings = pls.x_loadings_
elif hasattr(pls, '_x_mean'):
    pls_x_mean = pls._x_mean; pls_x_loadings = pls.x_loadings_
else:
    pls_x_mean     = np.load(CACHE('pls_mean.npy'))
    pls_x_loadings = np.load(CACHE('pls_loadings_T.npy')).T

PLS_LOADINGS_T  = torch.tensor(pls_x_loadings.T,  dtype=torch.float32).to(DEVICE)
PLS_MEAN        = torch.tensor(pls_x_mean,          dtype=torch.float32).to(DEVICE)
PLS_SCALER_STD  = torch.tensor(pls_scaler.scale_,   dtype=torch.float32).to(DEVICE)
PLS_SCALER_MEAN = torch.tensor(pls_scaler.mean_,    dtype=torch.float32).to(DEVICE)

def inverse_pls_torch(x_pls_scaled):
    x_raw = x_pls_scaled * PLS_SCALER_STD + PLS_SCALER_MEAN
    return x_raw @ PLS_LOADINGS_T + PLS_MEAN

print(f'Reloaded: PLS_DIM={PLS_DIM}  FW_DIM={FW_DIM}')
print(f'Train: {len(train_idx)}  Test: {len(test_idx)}')

Reloaded: PLS_DIM=10  FW_DIM=480
Train: 196  Test: 50


# ─────────────────────────────────────────────────────────────────────────────
# Section 2 — Model definitions
# ─────────────────────────────────────────────────────────────────────────────

In [7]:
class SinusoidalTimeEmbed(nn.Module):
    def __init__(self, dim=32):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        t    = t.view(-1, 1).float()
        half = self.dim // 2
        freqs = torch.exp(-np.log(10000) *
                torch.arange(half, device=t.device).float() / (half - 1))
        args = t * freqs.unsqueeze(0)
        return torch.cat([args.sin(), args.cos()], dim=-1)

class CDR3VelocityNetPLS(nn.Module):
    def __init__(self, pls_dim=10, fw_dim=480, dev_dim=2,
                 hidden=128, t_dim=32, drop=0.2):
        super().__init__()
        self.t_emb    = SinusoidalTimeEmbed(t_dim)
        self.fw_proj  = nn.Sequential(
            nn.Linear(fw_dim, hidden), nn.GELU(),
            nn.Linear(hidden, hidden * 2))
        self.dev_proj = nn.Sequential(
            nn.Linear(dev_dim, 32), nn.GELU(), nn.Linear(32, hidden))
        inp       = pls_dim + t_dim
        self.l1   = nn.Linear(inp, hidden)
        self.n1   = nn.LayerNorm(hidden)
        self.l2   = nn.Linear(hidden, hidden)
        self.n2   = nn.LayerNorm(hidden)
        self.out  = nn.Linear(hidden, pls_dim)
        self.act  = nn.GELU()
        self.drop = nn.Dropout(drop)
    def forward(self, t, x, fw, dev_target):
        if t.dim() == 0: t = t.expand(x.shape[0])
        t_e            = self.t_emb(t)
        h              = torch.cat([x, t_e], dim=-1)
        fw_scale, fw_b = self.fw_proj(fw).chunk(2, dim=-1)
        dev_e          = self.dev_proj(dev_target)
        h  = self.drop(self.act(self.n1(self.l1(h))))
        h  = h * (1 + fw_scale) + fw_b
        h  = h + dev_e
        h  = self.drop(self.act(self.n2(self.l2(h))))
        return self.out(h)

# ── CrossAttentionBindingScorer (REVISED: ab_proj, no contrastive) ────────────
class CrossAttentionBindingScorer(nn.Module):
    def __init__(self, ab_dim=480, ag_dim=ESM2_DIM,
                 hidden=SCORER_HIDDEN, n_heads=SCORER_HEADS, dropout=0.1):
        super().__init__()
        self.ab_proj    = nn.Sequential(
            nn.Linear(ab_dim, hidden, bias=False), nn.LayerNorm(hidden))
        self.ag_proj    = nn.Sequential(
            nn.Linear(ag_dim, hidden, bias=False), nn.LayerNorm(hidden))
        self.cross_attn = nn.MultiheadAttention(
            hidden, n_heads, dropout=dropout, batch_first=True)
        self.score_head = nn.Sequential(
            nn.Linear(hidden * 2, hidden), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden, 1))

    def forward_score(self, ab_emb, ag_residues, ag_key_padding_mask=None):
        q          = self.ab_proj(ab_emb).unsqueeze(1)
        kv         = self.ag_proj(ag_residues)
        context, _ = self.cross_attn(q, kv, kv,
                                     key_padding_mask=ag_key_padding_mask)
        combined   = torch.cat([q.squeeze(1), context.squeeze(1)], dim=-1)
        return self.score_head(combined).squeeze(-1)

print('Model classes defined.')

Model classes defined.


In [8]:
# ── Load flow model ───────────────────────────────────────────────────────────
FLOW_CKPT = CACHE(f'flow_model_pls_sigma{CURRENT_SIGMA}.pt')
flow_model = CDR3VelocityNetPLS(pls_dim=PLS_DIM, fw_dim=FW_DIM).to(DEVICE)
flow_model.load_state_dict(torch.load(FLOW_CKPT, map_location=DEVICE))
flow_model.eval()
print(f'Flow model loaded from: {FLOW_CKPT}')

# ── Load binding scorer (fine-tuned Stage 2 weights from Notebook 3) ──────────
binding_scorer = CrossAttentionBindingScorer().to(DEVICE)
SCORER_CKPT = f'{EVAL_DIR}/binding_scorer.pt'
binding_scorer.load_state_dict(torch.load(SCORER_CKPT, map_location=DEVICE))
binding_scorer.eval()
print(f'Binding scorer loaded from: {SCORER_CKPT}')

# ── Load AbLang2 ──────────────────────────────────────────────────────────────
ablang = ablang2.pretrained(model_to_use='ablang2-paired',
                            random_init=False, ncpu=1, device=str(DEVICE))
ablang.freeze()
print('AbLang2 loaded.')

# ── Load antigen embeddings ───────────────────────────────────────────────────
mean_embs      = np.load(MEAN_CACHE)
resid_embs     = np.load(RESID_CACHE)
valid_mask     = np.load(MASK_CACHE).astype(bool)
resid_pad_mask = np.load(RESID_PAD_CACHE).astype(bool)
print(f'Antigen embs loaded.  mean: {mean_embs.shape}  resid: {resid_embs.shape}')
print(f'  valid: {valid_mask.sum()} / {len(valid_mask)}')

Flow model loaded from: /content/drive/MyDrive/2026 Spring/BMI 702/project/flow_model_pls_sigma0.5.pt
Binding scorer loaded from: /content/drive/MyDrive/2026 Spring/BMI 702/project/eval_inputs/binding_scorer.pt
AbLang2 loaded.
Antigen embs loaded.  mean: (246, 1280)  resid: (246, 512, 1280)
  valid: 244 / 246


In [9]:
# ── Helper functions ──────────────────────────────────────────────────────────

@torch.no_grad()
def get_ab_seqcoding(vh, vl):
    """AbLang2 seqcoding of full (VH, VL) → 480d numpy vector."""
    emb = ablang([(str(vh), str(vl))], mode='seqcoding')
    return np.array(emb).reshape(480)

def get_pls_position(ab_emb_480):
    """Project 480d seqcoding → scaled PLS space (1, PLS_DIM)."""
    return pls_scaler.transform(pls.transform(ab_emb_480.reshape(1, -1)))

@torch.no_grad()
def get_binding_score(ab_emb_480, ag_resid_t, ag_mask_t):
    """Predict −log_Aff binding score using cross-attention scorer. Returns float."""
    ab_t = torch.tensor(ab_emb_480, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    return float(binding_scorer.forward_score(
        ab_t, ag_resid_t.unsqueeze(0), ag_mask_t.unsqueeze(0)).cpu())

def oracle_score(vh_seq, vl_seq):
    """Ridge oracle: full seqcoding → HIC + AC-SINS predictions."""
    emb = get_ab_seqcoding(vh_seq, vl_seq)
    emb_sc = full_scaler.transform(emb.reshape(1, -1))
    return {
        'hic':  float(oracle_hic.predict(emb_sc)[0]),
        'sins': float(oracle_sins.predict(emb_sc)[0]),
    }

def joint_score_zsum(sc_dict, binding_score):
    """Composite score for population selection. Lower = better."""
    zh = (sc_dict['hic']  - y_hic_train.mean())  / (y_hic_train.std()  + 1e-8)
    zs = (sc_dict['sins'] - y_sins_train.mean()) / (y_sins_train.std() + 1e-8)
    return zh + zs - binding_score

def splice_cdr3(parent_vh, cdr3_idx, new_cdr3):
    """Replace residues at cdr3_idx in parent_vh with new_cdr3."""
    vh = list(parent_vh)
    for i, pos in enumerate(cdr3_idx):
        if i < len(new_cdr3) and pos < len(vh):
            vh[pos] = new_cdr3[i]
    return ''.join(vh)

def mask_cdr3(vh_seq, cdr3_idx):
    """Replace CDR3 positions with AbLang2 mask token before forward pass."""
    chars = list(vh_seq)
    for i in cdr3_idx:
        if i < len(chars): chars[i] = ABLANG_MASK
    return ''.join(chars)

print('Helper functions defined.')

Helper functions defined.


In [10]:
# ── Build per-residue antigen tensors for test set ────────────────────────────
ag_resid_tensors, ag_pad_mask_tensors = {}, {}
for global_i in test_idx:
    if global_i < len(valid_mask) and valid_mask[global_i]:
        ag_resid_tensors[global_i]    = torch.tensor(
            resid_embs[global_i],     dtype=torch.float32).to(DEVICE)
        ag_pad_mask_tensors[global_i] = torch.tensor(
            resid_pad_mask[global_i], dtype=torch.bool).to(DEVICE)
print(f'Test antibodies: {len(test_idx)}')

Test antibodies: 50


# ─────────────────────────────────────────────────────────────────────────────
# Section 3 — Phase 1: Guided ODE Sampling
# Guidance uses forward_score (cross-attention), not forward_contrastive.
# ─────────────────────────────────────────────────────────────────────────────

In [11]:
def euler_integrate_guided(
        flow_model, binding_scorer,
        x0, fw, dev, ag_residues, ag_pad_mask,
        n_steps=N_ODE_STEPS, guidance_lambda=GUIDANCE_LAMBDA, device=DEVICE):
    """
    Euler integration with optional binding gradient guidance.
    Guidance: ∇_x forward_score(inverse_pls(x), ag) — cross-attention scorer.
    """
    x  = x0.clone()
    dt = 1.0 / n_steps
    flow_model.eval(); binding_scorer.eval()
    for t_scalar in torch.linspace(0.0, 1.0 - dt, n_steps, device=device):
        t_batch = t_scalar.expand(x.shape[0])
        with torch.no_grad():
            v_flow = flow_model(t_batch, x, fw, dev)

        if guidance_lambda > 0 and ag_residues is not None:
            x_g    = x.detach().requires_grad_(True)
            ab_480 = inverse_pls_torch(x_g)         # (B, 480) full seqcoding
            score  = binding_scorer.forward_score(
                ab_480, ag_residues, ag_pad_mask)    # cross-attention
            grad   = torch.autograd.grad(score.sum(), x_g, create_graph=False)[0]
            grad   = torch.nan_to_num(grad, nan=0.0, posinf=0.0, neginf=0.0)
            grad   = grad / grad.norm(dim=-1, keepdim=True).clamp(min=1e-8)
            v_total = v_flow + guidance_lambda * grad
        else:
            v_total = v_flow

        x = x.detach() + dt * v_total.detach()
    return x

print('euler_integrate_guided() defined.')

euler_integrate_guided() defined.


In [12]:
valid_test_local = [
    i for i in range(len(test_idx))
    if isinstance(df['hcdr3_sequence'].values[test_idx[i]], str)
    and len(df['hcdr3_sequence'].values[test_idx[i]]) > 0]
print(f'Valid test antibodies: {len(valid_test_local)}')

target_hic  = np.percentile(y_hic_train,  20)
target_sins = np.percentile(y_sins_train, 20)
dev_target_sc = dev_scaler.transform(np.array([[target_hic, target_sins]]))
print(f'Dev target: HIC={target_hic:.3f}  SINS={target_sins:.3f}')

# ── Phase 1: guided ODE (lambda = GUIDANCE_LAMBDA) ───────────────────────────
flow_targets = {}
print(f'Running Phase 1 (guided ODE, lambda={GUIDANCE_LAMBDA})...')
for local_i in valid_test_local:
    global_i = test_idx[local_i]
    ab_name  = df.iloc[global_i].get(COL_NAME, f'Ab_{global_i}')
    fw_cond  = torch.tensor(X_fw_te[[local_i]], dtype=torch.float32).to(DEVICE)
    dev_t    = torch.tensor(dev_target_sc,      dtype=torch.float32).to(DEVICE)
    x0       = torch.randn(N_FLOW_SAMPLES, PLS_DIM, device=DEVICE)

    has_ag = global_i in ag_resid_tensors
    if has_ag and GUIDANCE_LAMBDA > 0:
        ag_res  = ag_resid_tensors[global_i].unsqueeze(0).expand(N_FLOW_SAMPLES,-1,-1)
        ag_mask = ag_pad_mask_tensors[global_i].unsqueeze(0).expand(N_FLOW_SAMPLES,-1)
        lam = GUIDANCE_LAMBDA
    else:
        ag_res  = torch.zeros(N_FLOW_SAMPLES, 1, ESM2_DIM, device=DEVICE)
        ag_mask = torch.ones(N_FLOW_SAMPLES, 1, dtype=torch.bool, device=DEVICE)
        lam = 0.0
        if not has_ag:
            print(f'  No antigen for {ab_name}, running unguided')

    targets = euler_integrate_guided(
        flow_model, binding_scorer, x0,
        fw_cond.expand(N_FLOW_SAMPLES,-1), dev_t.expand(N_FLOW_SAMPLES,-1),
        ag_res, ag_mask, guidance_lambda=lam)
    flow_targets[ab_name] = targets.cpu().numpy()

print(f'Phase 1 complete: {len(flow_targets)} x {N_FLOW_SAMPLES} targets')

# ── Phase 1b: flow-only ODE (lambda = 0) ─────────────────────────────────────
flow_only_targets = {}
print('Running Phase 1b (flow-only ODE, lambda=0)...')
for local_i in valid_test_local:
    global_i = test_idx[local_i]
    ab_name  = df.iloc[global_i].get(COL_NAME, f'Ab_{global_i}')
    fw_cond  = torch.tensor(X_fw_te[[local_i]], dtype=torch.float32).to(DEVICE)
    dev_t    = torch.tensor(dev_target_sc,      dtype=torch.float32).to(DEVICE)
    x0       = torch.randn(N_FLOW_SAMPLES, PLS_DIM, device=DEVICE)
    ag_res   = torch.zeros(N_FLOW_SAMPLES, 1, ESM2_DIM, device=DEVICE)
    ag_mask  = torch.ones(N_FLOW_SAMPLES, 1, dtype=torch.bool, device=DEVICE)
    targets  = euler_integrate_guided(
        flow_model, binding_scorer, x0,
        fw_cond.expand(N_FLOW_SAMPLES,-1), dev_t.expand(N_FLOW_SAMPLES,-1),
        ag_res, ag_mask, guidance_lambda=0.0)
    flow_only_targets[ab_name] = targets.cpu().numpy()
    print(f'  ✓ {ab_name}')
print(f'Phase 1b complete: {len(flow_only_targets)} antibodies')

Valid test antibodies: 50
Dev target: HIC=2.560  SINS=-0.375
Running Phase 1 (guided ODE, lambda=0.5)...
  No antigen for racotumomab, running unguided
Phase 1 complete: 50 x 50 targets
Running Phase 1b (flow-only ODE, lambda=0)...
  ✓ abagovomab
  ✓ abituzumab
  ✓ abrilumab
  ✓ amatuximab
  ✓ basiliximab
  ✓ bavituximab
  ✓ belantamab
  ✓ bemarituzumab
  ✓ benralizumab
  ✓ blosozumab
  ✓ bococizumab
  ✓ brontictuzumab
  ✓ burosumab
  ✓ coltuximab
  ✓ crizanlizumab
  ✓ daclizumab
  ✓ eculizumab
  ✓ emactuzumab
  ✓ emibetuzumab
  ✓ enokizumab
  ✓ epratuzumab
  ✓ fasinumab
  ✓ fletikumab
  ✓ galcanezumab
  ✓ gemtuzumab
  ✓ imgatuzumab
  ✓ inotuzumab
  ✓ lampalizumab
  ✓ lintuzumab
  ✓ loncastuximab
  ✓ matuzumab
  ✓ mirikizumab
  ✓ monalizumab
  ✓ muromonab
  ✓ natalizumab
  ✓ nimotuzumab
  ✓ obiltoxaximab
  ✓ omburtamab
  ✓ pidilizumab
  ✓ racotumomab
  ✓ risankizumab
  ✓ rituximab
  ✓ romosozumab
  ✓ selicrelumab
  ✓ sintilimab
  ✓ tildrakizumab
  ✓ ublituximab
  ✓ vadastuximab
  ✓ vel

# ─────────────────────────────────────────────────────────────────────────────
# Section 4 — Phase 2: Steered AbLang2 Sampling (Iterative, 5 rounds)
# ─────────────────────────────────────────────────────────────────────────────

In [13]:
def get_cdr3_mlm_logits(ablang, masked_vh, vl_seq, cdr3_positions):
    """
    Single AbLang2 masked forward pass.
    Returns (L, 20) log-softmax probabilities over standard AAs at CDR3 positions.
    """
    with torch.no_grad():
        reps = ablang([(masked_vh, vl_seq)], mode='rescoding')
    hidden = np.array(reps[0])[:len(masked_vh)]
    L = len(cdr3_positions)
    logits = np.zeros((L, 20), dtype=np.float32)
    for k, pos in enumerate(cdr3_positions):
        if pos < len(hidden):
            logits[k] = hidden[pos, :20]  # first 20 dims = AA vocab
    # Convert to log-softmax
    from scipy.special import log_softmax
    return np.vstack([log_softmax(logits[k]) for k in range(L)])

def temperature_sample_sequences(log_probs, M=20, temperature=1.0, seed=None):
    """Sample M CDR3 sequences independently from (L, 20) log-prob distribution."""
    rng = np.random.default_rng(seed)
    L = log_probs.shape[0]
    seqs = []
    for _ in range(M):
        cdr3 = ''
        for pos in range(L):
            probs = np.exp(log_probs[pos] / temperature)
            probs = np.clip(probs, 0, None); probs /= probs.sum()
            cdr3 += AA_VOCAB[rng.choice(20, p=probs)]
        seqs.append(cdr3)
    return seqs

print('MLM helpers defined.')

MLM helpers defined.


In [14]:
def steer_logits(log_probs, flow_target_pls,
                 parent_vh, parent_vl, cdr3_idx,
                 ag_resid_t, ag_mask_t,
                 alpha=ALPHA_DEV, beta=BETA_BIND,
                 guidance_strength=GUIDANCE_STRENGTH,
                 top_k=TOP_K_TOKENS):
    """
    For each CDR3 position, evaluate top_k trial amino acids:
      1. Splice trial token into greedy CDR3
      2. get_ab_seqcoding(trial_vh, parent_vl)   ← ONE call, shared
      3. PLS distance to flow_target_pls          ← developability signal
      4. forward_score on that embedding           ← binding signal
      5. bonus = -alpha * pls_dist + beta * bind_score

    ag_resid_t : (L_ag, 1280) tensor on DEVICE, or None (no binding term)
    ag_mask_t  : (L_ag,) bool tensor on DEVICE, or None
    """
    L      = log_probs.shape[0]
    greedy = [AA_VOCAB[log_probs[pos].argmax()] for pos in range(L)]
    steered = log_probs.copy()

    for pos in range(L):
        base_probs = np.exp(log_probs[pos] / TEMPERATURE)
        base_probs = np.clip(base_probs, 0, None); base_probs /= base_probs.sum()
        top_k_idx  = np.argsort(base_probs)[-top_k:]

        bonus = np.zeros(20)
        for aa_idx in top_k_idx:
            trial      = greedy.copy()
            trial[pos] = AA_VOCAB[aa_idx]
            trial_vh   = splice_cdr3(parent_vh, cdr3_idx, ''.join(trial))

            # ONE seqcoding call — shared for PLS distance and binding
            ab_emb = get_ab_seqcoding(trial_vh, parent_vl)

            # Developability: distance to flow PLS target
            pls_pos = get_pls_position(ab_emb)
            dist    = float(np.linalg.norm(pls_pos - flow_target_pls))

            # Binding (skip if no antigen)
            bs = (get_binding_score(ab_emb, ag_resid_t, ag_mask_t)
                  if ag_resid_t is not None else 0.0)

            bonus[aa_idx] = -alpha * dist + beta * bs

        # Numerical stability + apply only to evaluated tokens
        evaluated = bonus[top_k_idx]
        bonus[top_k_idx] -= evaluated.max()
        weight = np.exp(guidance_strength * bonus)
        unevaluated = np.ones(20, dtype=bool); unevaluated[top_k_idx] = False
        weight[unevaluated] = 0.0

        combined = base_probs * (1.0 + weight)
        combined /= combined.sum()
        steered[pos] = np.log(combined + 1e-12)

    return steered

print('steer_logits() defined.')

steer_logits() defined.


In [15]:
def sample_with_binding_threshold(steered_log_probs,
                                   parent_vh, parent_vl, cdr3_idx,
                                   ag_resid_t, ag_mask_t,
                                   parent_binding_score, binding_threshold,
                                   M=20, max_tries=5):
    """Sample M sequences, reject those below binding_threshold. Fallback to best."""
    accepted, all_scored, tries = [], [], 0

    while len(accepted) < M and tries < max_tries:
        candidates = temperature_sample_sequences(steered_log_probs, M=M)
        for cdr3 in candidates:
            vh     = splice_cdr3(parent_vh, cdr3_idx, cdr3)
            ab_emb = get_ab_seqcoding(vh, parent_vl)
            bs     = (get_binding_score(ab_emb, ag_resid_t, ag_mask_t)
                      if ag_resid_t is not None else 0.0)
            sc     = oracle_score(vh, parent_vl)
            entry  = (cdr3, vh, ab_emb, bs, sc)
            all_scored.append(entry)
            if bs >= binding_threshold:
                accepted.append(entry)
            if len(accepted) >= M:
                break
        tries += 1

    # Fallback: fill remaining with highest binding scores
    if len(accepted) < M:
        all_scored.sort(key=lambda x: -x[3])
        seen = set(id(x) for x in accepted)
        accepted += [x for x in all_scored if id(x) not in seen][:M - len(accepted)]

    return accepted[:M]

print('sample_with_binding_threshold() defined.')

sample_with_binding_threshold() defined.


In [17]:
# ── Phase 2: Iterative steered sampling (5 rounds per antibody) ───────────────
print(f'Running Phase 2 (steered, {N_STEER_ROUNDS} rounds) for {len(valid_test_local)} antibodies...')
generation_records = []

for local_i in valid_test_local:
    global_i    = test_idx[local_i]
    row         = df.iloc[global_i]
    ab_name     = row.get(COL_NAME, f'Ab_{global_i}')
    parent_vh   = row[COL_VH]
    parent_vl   = row[COL_VL]
    cdr3_idx    = row['h_cdr3_idx']
    parent_cdr3 = row['hcdr3_sequence']

    has_antigen = global_i in ag_resid_tensors
    ag_r = ag_resid_tensors.get(global_i, None)
    ag_m = ag_pad_mask_tensors.get(global_i, None)

    # Parent binding score (computed once, used for threshold)
    parent_ab_emb  = get_ab_seqcoding(parent_vh, parent_vl)
    parent_binding = get_binding_score(parent_ab_emb, ag_r, ag_m) if has_antigen else 0.0

    # ── GUIDED condition ──────────────────────────────────────────────────────
    masked_vh  = mask_cdr3(parent_vh, cdr3_idx)
    log_probs  = get_cdr3_mlm_logits(ablang, masked_vh, parent_vl, cdr3_idx)
    # Round 1 centroid = ODE flow target mean
    current_centroid = flow_targets[ab_name].mean(0, keepdims=True)  # (1, PLS_DIM)
    population = []  # list of (cdr3, vh, ab_emb, bs, sc)
    best_overall = None; best_joint = float('inf')

    for round_idx in range(N_STEER_ROUNDS):
        slack     = BIND_SLACK_INIT * (1 - round_idx / N_STEER_ROUNDS)
        threshold = parent_binding - slack # when steer, only sample ones with higher binding prediction score than parent

        steered_lp = steer_logits(
            log_probs, current_centroid, parent_vh, parent_vl, cdr3_idx,
            ag_r if has_antigen else None, ag_m if has_antigen else None)

        candidates = sample_with_binding_threshold(
            steered_lp, parent_vh, parent_vl, cdr3_idx,
            ag_r, ag_m, parent_binding, threshold,
            M=M_SEQUENCES)

        # Score + select top-k by joint score
        scored = []
        for (cdr3, vh, ab_emb, bs, sc) in candidates:
            js = joint_score_zsum(sc, bs)
            scored.append((js, cdr3, vh, ab_emb, bs, sc))
            if js < best_joint:
                best_joint = js; best_overall = (cdr3, vh, bs, sc)

        scored.sort(key=lambda x: x[0])
        population = scored[:TOP_K_STEER]

        # Update centroid from top-k population seqcodings
        pop_embs = np.vstack([e[3] for e in population])  # (k, 480)
        pop_pls  = pls_scaler.transform(pls.transform(pop_embs))  # (k, PLS_DIM)
        current_centroid = pop_pls.mean(0, keepdims=True)

    guided_cdr3 = best_overall[0]
    guided_vh   = best_overall[1]
    guided_bs   = best_overall[2]

    # ── FLOW-ONLY condition ───────────────────────────────────────────────────
    fo_centroid = flow_only_targets[ab_name].mean(0, keepdims=True)
    fo_best = None; fo_best_joint = float('inf')

    for round_idx in range(N_STEER_ROUNDS):
        steered_lp = steer_logits(
            log_probs, fo_centroid, parent_vh, parent_vl, cdr3_idx,
            None, None, beta=0.0)

        # No binding threshold — sample freely, score by oracle only
        candidates = temperature_sample_sequences(steered_lp, M=M_SEQUENCES)

        scored = []
        for cdr3 in candidates:
            vh     = splice_cdr3(parent_vh, cdr3_idx, cdr3)
            ab_emb = get_ab_seqcoding(vh, parent_vl)
            bs     = get_binding_score(ab_emb, ag_r, ag_m) if has_antigen else 0.0
            sc     = oracle_score(vh, parent_vl)
            js     = joint_score_zsum(sc, bs)
            scored.append((js, cdr3, vh, ab_emb, bs, sc))
            if js < fo_best_joint:
                fo_best_joint = js; fo_best = (cdr3, vh, bs, sc)

        scored.sort(key=lambda x: x[0])
        fo_population = scored[:TOP_K_STEER]
        pop_embs  = np.vstack([e[3] for e in fo_population])
        pop_pls   = pls_scaler.transform(pls.transform(pop_embs))
        fo_centroid = pop_pls.mean(0, keepdims=True)

    flow_only_cdr3 = fo_best[0]
    flow_only_vh   = fo_best[1]
    flow_only_bs   = fo_best[2]

    generation_records.append({
        'ab_name':     ab_name,
        'global_idx':  global_i,
        'local_idx':   local_i,
        'parent_vh':   parent_vh,
        'parent_vl':   parent_vl,
        'parent_cdr3': parent_cdr3,
        'cdr3_idx':    cdr3_idx,
        'guided_cdr3': guided_cdr3,
        'guided_vh':   guided_vh,
        'guided_bs':   guided_bs,
        'flow_only_cdr3': flow_only_cdr3,
        'flow_only_vh':   flow_only_vh,
        'flow_only_bs':   flow_only_bs,
        'parent_binding': parent_binding,
    })
    print(f'  ✓ {ab_name}  guided_bs={guided_bs:.3f}  fo_bs={flow_only_bs:.3f}')

print(f'\nPhase 2 complete: {len(generation_records)} antibodies.')

Running Phase 2 (steered, 5 rounds) for 50 antibodies...
  ✓ abagovomab  guided_bs=-1.080  fo_bs=-1.026
  ✓ abituzumab  guided_bs=-0.240  fo_bs=-0.115
  ✓ abrilumab  guided_bs=-1.028  fo_bs=-1.083
  ✓ amatuximab  guided_bs=-1.179  fo_bs=-1.162
  ✓ basiliximab  guided_bs=-1.452  fo_bs=-1.494
  ✓ bavituximab  guided_bs=-1.480  fo_bs=-1.433
  ✓ belantamab  guided_bs=-0.541  fo_bs=-0.709
  ✓ bemarituzumab  guided_bs=-0.533  fo_bs=-0.562
  ✓ benralizumab  guided_bs=-1.057  fo_bs=-1.234
  ✓ blosozumab  guided_bs=-0.692  fo_bs=-0.790
  ✓ bococizumab  guided_bs=-1.102  fo_bs=-1.259
  ✓ brontictuzumab  guided_bs=-0.697  fo_bs=-0.649
  ✓ burosumab  guided_bs=-1.413  fo_bs=-1.421
  ✓ coltuximab  guided_bs=-1.026  fo_bs=-0.977
  ✓ crizanlizumab  guided_bs=-0.657  fo_bs=-0.629
  ✓ daclizumab  guided_bs=-0.854  fo_bs=-0.996
  ✓ eculizumab  guided_bs=-0.846  fo_bs=-0.786
  ✓ emactuzumab  guided_bs=-1.172  fo_bs=-1.206
  ✓ emibetuzumab  guided_bs=-0.821  fo_bs=-0.746
  ✓ enokizumab  guided_bs=-1.606  

# ─────────────────────────────────────────────────────────────────────────────
# Section 5 — Oracle Scoring + AbLang2-Only Baseline
# ─────────────────────────────────────────────────────────────────────────────

In [18]:
# ── Oracle scoring for guided + flow-only ─────────────────────────────────────
results = []
for rec in generation_records:
    ab_name = rec['ab_name']
    par = oracle_score(rec['parent_vh'], rec['parent_vl'])
    gui = oracle_score(rec['guided_vh'], rec['parent_vl'])
    fo  = oracle_score(rec['flow_only_vh'], rec['parent_vl'])

    results.append({
        'ab_name':               ab_name,
        'parent_vh':             rec['parent_vh'],
        'parent_vl':             rec['parent_vl'],
        'parent_cdr3':           rec['parent_cdr3'],
        'guided_cdr3':           rec['guided_cdr3'],
        'guided_vh':             rec['guided_vh'],
        'flow_only_cdr3':        rec['flow_only_cdr3'],
        'flow_only_vh':          rec['flow_only_vh'],
        'oracle_hic_parent':     par['hic'],
        'oracle_sins_parent':    par['sins'],
        'oracle_hic_guided':     gui['hic'],
        'oracle_sins_guided':    gui['sins'],
        'oracle_hic_flow_only':  fo['hic'],
        'oracle_sins_flow_only': fo['sins'],
        'binding_parent':        rec['parent_binding'],
        'binding_guided':        rec['guided_bs'],
        'binding_flow_only':     rec['flow_only_bs'],
        'h_cdr3_idx':            rec['cdr3_idx'],
    })
    print(f'  ✓ {ab_name}  '
          f'HIC: {par["hic"]:+.3f}→{gui["hic"]:+.3f} (guided)  '
          f'SINS: {par["sins"]:+.3f}→{gui["sins"]:+.3f}')

seq_df = pd.DataFrame(results)
seq_df.to_csv(CACHE('seq_df.csv'), index=False)
print(f'\nSaved {len(seq_df)} results.')

  ✓ abagovomab  HIC: +3.083→+3.096 (guided)  SINS: +19.075→+10.224
  ✓ abituzumab  HIC: +2.705→+2.295 (guided)  SINS: +8.425→+5.017
  ✓ abrilumab  HIC: +2.745→+1.887 (guided)  SINS: -10.797→-19.379
  ✓ amatuximab  HIC: +2.639→+2.631 (guided)  SINS: +4.656→+1.636
  ✓ basiliximab  HIC: +2.934→+2.429 (guided)  SINS: +20.353→+13.281
  ✓ bavituximab  HIC: +2.309→+1.925 (guided)  SINS: +2.422→-5.590
  ✓ belantamab  HIC: +2.960→+2.807 (guided)  SINS: +11.020→-2.484
  ✓ bemarituzumab  HIC: +2.438→+2.158 (guided)  SINS: -1.524→-7.766
  ✓ benralizumab  HIC: +2.551→+2.436 (guided)  SINS: -3.640→-11.858
  ✓ blosozumab  HIC: +2.798→+2.573 (guided)  SINS: -4.587→-6.678
  ✓ bococizumab  HIC: +2.125→+1.850 (guided)  SINS: +15.609→+13.395
  ✓ brontictuzumab  HIC: +3.286→+2.851 (guided)  SINS: +5.335→+3.679
  ✓ burosumab  HIC: +3.017→+2.807 (guided)  SINS: +0.960→-14.402
  ✓ coltuximab  HIC: +2.723→+2.406 (guided)  SINS: -0.126→-5.628
  ✓ crizanlizumab  HIC: +2.490→+2.320 (guided)  SINS: -8.579→-13.777


In [19]:
# ── AbLang2-only baseline ─────────────────────────────────────────────────────
baseline_records = []
print('Running AbLang2-only baseline (no flow model)...')
for local_i in valid_test_local:
    global_i  = test_idx[local_i]
    row       = df.iloc[global_i]
    ab_name   = row.get(COL_NAME, f'Ab_{global_i}')
    parent_vh = row[COL_VH]; parent_vl = row[COL_VL]
    cdr3_idx  = row['h_cdr3_idx']

    has_antigen = global_i in ag_resid_tensors
    ag_r = ag_resid_tensors.get(global_i, None)
    ag_m = ag_pad_mask_tensors.get(global_i, None)

    log_probs  = get_cdr3_mlm_logits(
        ablang, mask_cdr3(parent_vh, cdr3_idx), parent_vl, cdr3_idx)
    candidates = temperature_sample_sequences(log_probs, M=M_SEQUENCES,
                                              temperature=TEMPERATURE,
                                              seed=int(global_i))
    spliced    = [splice_cdr3(parent_vh, cdr3_idx, c) for c in candidates]

    best_joint, best_hic, best_sins, best_cdr3, best_vh, best_bs =         float('inf'), None, None, None, None, 0.0
    for cdr3, vh in zip(candidates, spliced):
        ab_emb = get_ab_seqcoding(vh, parent_vl)
        bs     = get_binding_score(ab_emb, ag_r, ag_m) if has_antigen else 0.0
        sc     = oracle_score(vh, parent_vl)
        js     = joint_score_zsum(sc, bs)
        if js < best_joint:
            best_joint = js; best_hic = sc['hic']; best_sins = sc['sins']
            best_cdr3  = cdr3; best_vh = vh; best_bs = bs

    baseline_records.append({
        'ab_name':              ab_name,
        'baseline_cdr3':        best_cdr3,
        'baseline_vh':          best_vh,     # needed for CamSol/SAP in eval
        'binding_baseline':     best_bs,
        'oracle_hic_baseline':  best_hic,
        'oracle_sins_baseline': best_sins,
    })
    print(f'  ✓ {ab_name}  HIC={best_hic:.3f}  SINS={best_sins:.3f}')

baseline_df = pd.DataFrame(baseline_records)

compare_df = seq_df.merge(
    baseline_df[['ab_name', 'oracle_hic_baseline', 'oracle_sins_baseline',
                 'baseline_cdr3', 'baseline_vh', 'binding_baseline']],
    on='ab_name', how='inner')
print(f'\ncomparison dataframe: {len(compare_df)} antibodies')
print(f'Columns: {compare_df.columns.tolist()}')

Running AbLang2-only baseline (no flow model)...
  ✓ abagovomab  HIC=2.864  SINS=19.615
  ✓ abituzumab  HIC=2.716  SINS=5.755
  ✓ abrilumab  HIC=1.899  SINS=-18.214
  ✓ amatuximab  HIC=2.543  SINS=-0.452
  ✓ basiliximab  HIC=2.622  SINS=15.338
  ✓ bavituximab  HIC=1.889  SINS=-5.281
  ✓ belantamab  HIC=2.667  SINS=8.162
  ✓ bemarituzumab  HIC=2.447  SINS=-11.451
  ✓ benralizumab  HIC=2.203  SINS=-2.589
  ✓ blosozumab  HIC=2.603  SINS=-8.327
  ✓ bococizumab  HIC=2.304  SINS=2.324
  ✓ brontictuzumab  HIC=2.963  SINS=4.436
  ✓ burosumab  HIC=2.857  SINS=-4.220
  ✓ coltuximab  HIC=2.687  SINS=-6.588
  ✓ crizanlizumab  HIC=2.294  SINS=-9.323
  ✓ daclizumab  HIC=2.800  SINS=-2.453
  ✓ eculizumab  HIC=2.662  SINS=0.627
  ✓ emactuzumab  HIC=2.270  SINS=-2.592
  ✓ emibetuzumab  HIC=2.434  SINS=1.655
  ✓ enokizumab  HIC=3.132  SINS=8.049
  ✓ epratuzumab  HIC=2.428  SINS=3.143
  ✓ fasinumab  HIC=2.480  SINS=-3.807
  ✓ fletikumab  HIC=2.634  SINS=-3.293
  ✓ galcanezumab  HIC=3.148  SINS=1.555
  ✓ 

In [20]:
# Summary print
NO_ANTIGEN_ABS = {'panobacumab', 'racotumomab'}
compare_df_binding = compare_df[
    ~compare_df['ab_name'].str.lower().isin(NO_ANTIGEN_ABS)
].reset_index(drop=True)

print(f'compare_df (all):           {len(compare_df)}')
print(f'compare_df_binding (w/ Ag): {len(compare_df_binding)}')

print(f'\n{"Condition":<28} {"HIC":>14} {"SINS":>14} {"Binding":>12}')
print('─' * 72)
for label, hc, sc_, bc in [
        ('Parent',             'oracle_hic_parent',   'oracle_sins_parent',   'binding_parent'),
        ('Baseline (AbLang2)', 'oracle_hic_baseline', 'oracle_sins_baseline', 'binding_baseline'),
        ('Flow-only',          'oracle_hic_flow_only','oracle_sins_flow_only','binding_flow_only'),
        ('Guided',             'oracle_hic_guided',   'oracle_sins_guided',   'binding_guided')]:
    h = compare_df_binding[hc]
    s = compare_df_binding[sc_]
    b = compare_df_binding[bc]
    print(f'{label:<28} {h.mean():>+6.3f}±{h.sem():.3f}  '
          f'{s.mean():>+6.3f}±{s.sem():.3f}  '
          f'{b.mean():>+8.3f}±{b.sem():.3f}')

compare_df (all):           50
compare_df_binding (w/ Ag): 49

Condition                               HIC           SINS      Binding
────────────────────────────────────────────────────────────────────────
Parent                       +2.732±0.049  +3.986±1.215    -0.897±0.055
Baseline (AbLang2)           +2.537±0.046  +0.031±1.167    -1.029±0.052
Flow-only                    +2.431±0.047  -2.155±1.166    -1.011±0.051
Guided                       +2.401±0.049  -1.285±1.209    -0.956±0.055


# ─────────────────────────────────────────────────────────────────────────────
# Section 6 — Save Artifacts for Evaluation Notebook
# ─────────────────────────────────────────────────────────────────────────────

In [21]:
from sklearn.decomposition import PCA

FULL_CACHE = CACHE('full_embeddings.npy')
CDR3_CACHE = CACHE('cdr3_embeddings.npy')

full_embs = np.load(FULL_CACHE) if os.path.exists(FULL_CACHE) else None
cdr3_embs = np.load(CDR3_CACHE) if os.path.exists(CDR3_CACHE) else None

if full_embs is None or cdr3_embs is None:
    print('WARNING: full_embeddings.npy or cdr3_embeddings.npy missing — '
          'please ensure Notebook 2 has been run.')
else:
    from sklearn.preprocessing import StandardScaler as _SS
    _fw_scaler   = _SS(); _cdr3_scaler = _SS()
    X_full_tr_ev = full_scaler.transform(full_embs[train_idx])
    X_full_te_ev = full_scaler.transform(full_embs[test_idx])
    X_cdr3_tr_ev = _cdr3_scaler.fit_transform(cdr3_embs[train_idx])
    X_cdr3_te_ev = _cdr3_scaler.transform(cdr3_embs[test_idx])

    pca = PCA(n_components=50, random_state=42).fit(X_cdr3_tr_ev)
    full_pca = PCA(n_components=20, random_state=42).fit(X_full_tr_ev)

    joblib.dump(pca,          f'{EVAL_DIR}/pca_cdr3.pkl')
    joblib.dump(full_pca,     f'{EVAL_DIR}/pca_full.pkl')
    joblib.dump(_cdr3_scaler, f'{EVAL_DIR}/cdr3_scaler.pkl')
    joblib.dump(_fw_scaler,   f'{EVAL_DIR}/fw_scaler.pkl')

    np.save(f'{EVAL_DIR}/X_cdr3_tr.npy', X_cdr3_tr_ev)
    np.save(f'{EVAL_DIR}/X_cdr3_te.npy', X_cdr3_te_ev)
    np.save(f'{EVAL_DIR}/X_fw_te.npy',   X_fw_te)
    np.save(f'{EVAL_DIR}/X_full_tr.npy', X_full_tr_ev)
    np.save(f'{EVAL_DIR}/X_full_te.npy', X_full_te_ev)
    np.save(f'{EVAL_DIR}/train_pc20.npy', pca.transform(cdr3_embs[train_idx])[:, :20])
    print('Saved PCA objects and embedding arrays.')

# DataFrames
compare_df.to_csv(f'{EVAL_DIR}/compare_df.csv', index=False)
seq_df.to_csv(    f'{EVAL_DIR}/seq_df.csv',     index=False)
compare_df_binding.to_csv(f'{EVAL_DIR}/compare_df_binding.csv', index=False)
print('Saved: compare_df.csv, seq_df.csv, compare_df_binding.csv')

# Original dataframe
df[['antibody_name','vh_protein_sequence','vl_protein_sequence',
    'hcdr3_sequence','hcdr3_len','hierarchical_cluster_fold',
    'liability_count','HIC','AC-SINS_pH7.4']
  ].to_csv(f'{EVAL_DIR}/df_sequences.csv', index=False)

# CDR3 index lists
cdr3_idx_dict = {int(i): df['h_cdr3_idx'].iloc[i]
                 for i in range(len(df))
                 if isinstance(df['h_cdr3_idx'].iloc[i], list)}
with open(f'{EVAL_DIR}/cdr3_idx.json', 'w') as fh:
    json.dump(cdr3_idx_dict, fh)
print('Saved: df_sequences.csv, cdr3_idx.json')

# Oracle models and scalers
joblib.dump(oracle_hic,   f'{EVAL_DIR}/oracle_hic.pkl')
joblib.dump(oracle_sins,  f'{EVAL_DIR}/oracle_sins.pkl')
joblib.dump(full_scaler,  f'{EVAL_DIR}/full_scaler.pkl')
joblib.dump(pls,          f'{EVAL_DIR}/pls_model.pkl')
joblib.dump(pls_scaler,   f'{EVAL_DIR}/pls_scaler.pkl')
joblib.dump(dev_scaler,   f'{EVAL_DIR}/dev_scaler.pkl')
print('Saved oracle models and scalers.')

# Oracle targets + split indices
def impute_train_median(df, col, train_idx):
    vals   = df[col].copy().astype(float)
    median = float(np.nanmedian(vals.values[train_idx]))
    return vals.fillna(median).values, median

y_hic,  _ = impute_train_median(df, COL_HIC,  train_idx)
y_sins, _ = impute_train_median(df, COL_SINS, train_idx)
y_hic_train_ev  = y_hic[train_idx];  y_hic_test_ev  = y_hic[test_idx]
y_sins_train_ev = y_sins[train_idx]; y_sins_test_ev = y_sins[test_idx]

np.save(f'{EVAL_DIR}/y_hic_train.npy',  y_hic_train_ev)
np.save(f'{EVAL_DIR}/y_sins_train.npy', y_sins_train_ev)
np.save(f'{EVAL_DIR}/y_hic_test.npy',   y_hic_test_ev)
np.save(f'{EVAL_DIR}/y_sins_test.npy',  y_sins_test_ev)
np.save(f'{EVAL_DIR}/train_idx.npy',    train_idx)
np.save(f'{EVAL_DIR}/test_idx.npy',     test_idx)

np.save(f'{EVAL_DIR}/pls_loadings_T.npy', pls_x_loadings.T.astype(np.float32))
np.save(f'{EVAL_DIR}/pls_mean.npy',       pls_x_mean.astype(np.float32))

print('Saved oracle targets, split indices, PLS tensors.')

# Inventory
print(f'\nAll artifacts saved to: {EVAL_DIR}')
for fname in sorted(os.listdir(EVAL_DIR)):
    sz = os.path.getsize(f'{EVAL_DIR}/{fname}') / 1024
    print(f'  {fname:<46} {sz:>8.1f} KB')
print('\nGeneration & Evaluation Complete. Proceed to evaluation notebook.')

Saved PCA objects and embedding arrays.
Saved: compare_df.csv, seq_df.csv, compare_df_binding.csv
Saved: df_sequences.csv, cdr3_idx.json
Saved oracle models and scalers.
Saved oracle targets, split indices, PLS tensors.

All artifacts saved to: /content/drive/MyDrive/2026 Spring/BMI 702/project/eval_inputs
  X_cdr3_te.npy                                      93.9 KB
  X_cdr3_tr.npy                                     367.6 KB
  X_full_te.npy                                     187.6 KB
  X_full_tr.npy                                     735.1 KB
  X_fw_te.npy                                        93.9 KB
  abrank_stage1.csv                              194491.1 KB
  abrank_stage2.csv                               19583.7 KB
  antigen_esm2_embs.npy                            1230.1 KB
  antigen_esm2_resid.npy                         629760.1 KB
  antigen_resid_pad_mask.npy                        123.1 KB
  antigen_valid_mask.npy                              0.4 KB
  binding_scorer.pt  